In [0]:
file_path = 'abfss://lendingclub@stlendingclubdev01.dfs.core.windows.net/bronze/customers_data'

In [0]:
customer_schema = 'member_id string, emp_title string, emp_length string, home_ownership string, annual_inc float, addr_state string, zip_code string, country string, grade string, sub_grade string, verification_status string, tot_hi_cred_lim float, application_type string, annual_inc_joint float, verification_status_joint string'

In [0]:
customers_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(customer_schema) \
.load("abfss://lendingclub@stlendingclubdev01.dfs.core.windows.net/bronze/customers_data")

In [0]:
customers_raw_df.display()

In [0]:
customers_raw_df.printSchema()

In [0]:
customer_df_renamed = customers_raw_df.withColumnRenamed("annual_inc", "annual_income") \
.withColumnRenamed("addr_state", "address_state") \
.withColumnRenamed("zip_code", "address_zipcode") \
.withColumnRenamed("country", "address_country") \
.withColumnRenamed("tot_hi_credit_lim", "total_high_credit_limit") \
.withColumnRenamed("annual_inc_joint", "join_annual_income")

In [0]:
customer_df_renamed.display()

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
customers_df_ingestd = customer_df_renamed.withColumn("ingest_date", current_timestamp())

In [0]:
customers_df_ingestd.count()

In [0]:
customers_distinct = customers_df_ingestd.distinct()
customers_distinct.count()


In [0]:
customers_distinct.count()


In [0]:
customers_distinct.createOrReplaceTempView("customers")

In [0]:
spark.sql("select * from customers").display()

In [0]:
spark.sql("select count(*) from customers where annual_income is null").display()

In [0]:
spark.sql("select distinct(emp_length) from customers").display()

In [0]:
spark.sql("select distinct(emp_length) from customers")

In [0]:
from pyspark.sql.functions import regexp_replace, col

In [0]:
customers_emplength_cleaned = customers_distinct.filter(col("annual_income").isNotNull()).withColumn("emp_length", regexp_replace(col("emp_length"), "(\\D)",""))

In [0]:
customers_emplength_cleaned.display()


In [0]:
customers_emplength_cleaned.printSchema()


In [0]:
customers_emplength_casted = customers_emplength_cleaned.withColumn("emp_length", customers_emplength_cleaned.emp_length.cast('int'))

In [0]:
customers_emplength_casted.display()

In [0]:
customers_emplength_casted.createOrReplaceTempView("customers")

In [0]:
%sql
select * from customers

In [0]:
spark.sql("select distinct(address_state) from customers")

In [0]:
%sql
select count(address_state) from customers where length(address_state)>2

In [0]:
from pyspark.sql.functions import when, col, length

In [0]:
customers_state_cleaned = customers_emplength_casted.withColumn(
    "address_state",
    when(length(col("address_state"))> 2, "NA").otherwise(col("address_state"))
)

In [0]:
customers_state_cleaned.display()


In [0]:
customers_state_cleaned.select("address_state").distinct()

In [0]:
customers_state_cleaned.write \
.format("delta") \
.mode("overwrite") \
.option("path", "abfss://lendingclub@stlendingclubdev01.dfs.core.windows.net/silver/customers") \
.save()